# Pooling (max / average)

_Deep Learning_

**Shrink the feature map by summarizing each little region with one number.**

Feature maps can be large. Pooling shrinks them to keep the important signal and cut the size.

     Max pooling takes the biggest number in each small region. Average pooling takes the average.

---

This notebook is a practice scaffold for the **Pooling (max / average)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns** — each sample is an 8x8 grid of pixel intensities (0–16).

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
print("image array:", digits.images.shape, " labels:", digits.target.shape)
samples = list(zip(digits.images, digits.target))
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — PyTorch

We build the two pooling layers and run a tiny 4x4 grid through them, so you can read off exactly which number each 2x2 window produces.

### Step 1 — Create the pooling layers and an input

A pooling layer slides a small window over the feature map and replaces each window with one summary number. Here both windows are 2x2 with stride 2, so they tile the grid without overlapping. The input is the numbers 0–15 laid out in a 4x4 grid; PyTorch expects a 4-D tensor shaped `(batch, channels, height, width)`, hence the leading `1, 1`.

In [ ]:
import torch
import torch.nn as nn

maxpool = nn.MaxPool2d(kernel_size=2, stride=2)   # 2x2 windows, no overlap
avgpool = nn.AvgPool2d(kernel_size=2, stride=2)

x = torch.arange(16, dtype=torch.float32).reshape(1, 1, 4, 4)
print("input 4x4:\n", x[0, 0])

### Step 2 — Max-pool: keep the strongest pixel

Max pooling reports the **largest** value inside each 2x2 block. The 4x4 grid has four such blocks, so the output is 2x2. Max pooling preserves the sharpest activations — useful when the *presence* of a strong feature matters more than its exact location.

In [ ]:
max_result = maxpool(x)
print("max-pool ->", tuple(max_result.shape))   # (1, 1, 2, 2)
print(max_result[0, 0])                          # the max of each 2x2 block

### Step 3 — Average-pool: smooth each block

Average pooling reports the **mean** of each 2x2 block instead of its maximum. It produces the same 2x2 shape but smooths the values, blending the whole region rather than spotlighting the single strongest pixel.

In [ ]:
avg_result = avgpool(x)
print("avg-pool ->")
print(avg_result[0, 0])   # the mean of each block

## Visualize the data & results

_How do max-pool and avg-pool shrink a real 8x8 digit image into a 4x4 summary?_

### Step 1 — Grab a real 8x8 image

Instead of the toy grid, we now pool a genuine handwritten digit. `digits.images[0]` is an 8x8 image of a zero — small enough to see every pixel, big enough to show what pooling throws away.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits

digits = load_digits()
img = digits.images[0]   # real 8x8 image of a 0

### Step 2 — Pool it down to 4x4 with NumPy

We reshape the 8x8 image into a `(4, 2, 4, 2)` array — a 4x4 grid of 2x2 blocks. Taking the max over the two 2-length axes gives max pooling; taking the mean over the same axes gives average pooling. Both collapse each 2x2 block to a single number, halving the height and width to 4x4.

In [ ]:
blocks = img.reshape(4, 2, 4, 2)   # split into a 4x4 grid of 2x2 blocks
maxp = blocks.max(axis=(1, 3))     # strongest pixel per block
avgp = blocks.mean(axis=(1, 3))    # mean per block

### Step 3 — Compare the three images side by side

Plotting the original next to its two pooled versions shows the trade-off: max-pool keeps the bright strokes crisp, while avg-pool produces a softer, blurrier summary. Both are half the size of the original yet still clearly a zero.

In [ ]:
fig, ax = plt.subplots(1, 3)
images = [img, maxp, avgp]
titles = ["Real 8x8 (a 0)", "Max-pool 2x2", "Avg-pool 2x2"]
for a, m, t in zip(ax, images, titles):
    a.imshow(m, cmap="viridis")
    a.set_title(t)
plt.show()